# 📚 Document Indexing with RAG

This notebook demonstrates how to upload and index documents into the ggssgs251
knowledge base using the RAG pipeline. You can use this notebook to batch-index
multiple documents programmatically, outside of the web UI.

## Prerequisites

- The backend server is running (`uvicorn backend.app:app --reload --port 8000`)
- Ollama is running with `nomic-embed-text` model pulled
- You have a valid JWT token (register/login first)

In [ ]:
import requests
import json
from pathlib import Path

# ── Configuration ──
API_BASE = "http://localhost:8000"

# Replace with your credentials
USERNAME = "testuser"
PASSWORD = "password123"

# Path to documents you want to index
DOCS_DIR = Path("../data/sample_docs")

## Step 1: Get JWT Token

In [ ]:
# Login to get a token
login_resp = requests.post(
    f"{API_BASE}/auth/login",
    json={"username": USERNAME, "password": PASSWORD},
)
login_resp.raise_for_status()
token = login_resp.json()["access_token"]
headers = {"Authorization": f"Bearer {token}"}

print(f"✅ Logged in as '{USERNAME}'")
print(f"📝 Token: {token[:30]}...")

## Step 2: List Currently Indexed Documents

In [ ]:
doc_resp = requests.get(f"{API_BASE}/rag/documents", headers=headers)
doc_resp.raise_for_status()
docs = doc_resp.json()

print(f"📚 Currently indexed: {docs['count']} documents")
for d in docs['documents']:
    print(f"   - {d['filename']}")

## Step 3: Upload and Index a Single Document

In [ ]:
def upload_document(filepath: str) -> dict:
    """Upload and index a document."""
    path = Path(filepath)
    if not path.exists():
        return {"error": f"File not found: {filepath}"}

    with open(path, "rb") as f:
        resp = requests.post(
            f"{API_BASE}/rag/upload",
            headers=headers,
            files={"file": (path.name, f, "application/octet-stream")},
        )

    if resp.status_code == 200:
        return resp.json()
    else:
        return {"error": f"HTTP {resp.status_code}: {resp.text}"}


# Example: Upload a text file
# result = upload_document("path/to/your/document.pdf")
# print(json.dumps(result, indent=2))

print("📄 To upload a document, call:")
print('    result = upload_document("path/to/file.pdf")')

## Step 4: Batch Index Multiple Documents

In [ ]:
def batch_upload(directory: str) -> list[dict]:
    """Upload all supported documents in a directory."""
    dir_path = Path(directory)
    supported = {".txt", ".md", ".pdf", ".docx"}
    results = []

    files = [f for f in dir_path.iterdir() if f.suffix.lower() in supported]
    print(f"📁 Found {len(files)} documents in '{directory}'")

    for file in files:
        print(f"   Uploading {file.name}...", end=" ")
        result = upload_document(str(file))
        results.append(result)
        if "error" in result:
            print(f"❌ {result['error']}")
        else:
            print(f"✅ {result['chunks']} chunks indexed")

    return results


# Example:
# results = batch_upload("../data/sample_docs")
# print(json.dumps(results, indent=2))

print("📁 To batch upload, call:")
print('    results = batch_upload("path/to/docs/dir")')

## Step 5: Delete a Document from the Index

In [ ]:
def delete_document(filename: str) -> dict:
    """Remove a document from the knowledge base."""
    import urllib.parse
    encoded = urllib.parse.quote(filename)
    resp = requests.delete(
        f"{API_BASE}/rag/documents/{encoded}",
        headers=headers,
    )
    if resp.status_code == 200:
        return resp.json()
    else:
        return {"error": f"HTTP {resp.status_code}: {resp.text}"}


print("🗑️ To delete a document, call:")
print('    result = delete_document("filename.pdf")')

## Step 6: Verify Index by Querying

In [ ]:
def query_knowledge_base(query: str, top_k: int = 5) -> dict:
    """Search the knowledge base."""
    resp = requests.post(
        f"{API_BASE}/rag/query",
        headers=headers,
        json={"query": query, "top_k": top_k},
    )
    resp.raise_for_status()
    return resp.json()


# Test query
test_query = "What topics are covered in our documents?"
print(f"🔍 Query: '{test_query}'")
print("-" * 50)

result = query_knowledge_base(test_query)
print(f"Found {len(result['sources'])} relevant passages")
for src in result['sources']:
    print(f"\n📄 From: {src['source']} (relevance: {src['score']:.2f})")
    print(f"   {src['content'][:150]}...")

## Troubleshooting

| Error | Solution |
|-------|----------|
| Connection refused | Ensure backend is running: `uvicorn backend.app:app --reload --port 8000` |
| 401 Unauthorized | Your token expired. Re-run the login cell. |
| 422 Guardrail blocked | Your query contains flagged content. Remove PII or profanity. |
| No chunks produced | The file may be empty or the text extraction failed. |
| Embedding failed | Ensure Ollama is running with `nomic-embed-text` model pulled. |